# 01. Data loading

This notebook prepares the dataset for all following experiments. It downloads the TACO archive, reads the original `annotations.json`, converts the COCO-like annotation structure into pandas dataframes, builds a single object-level dataframe, validates image paths, and saves the processed metadata for later training notebooks.

The output of this notebook is not a trained model. Its purpose is to make the dataset structure explicit and reproducible.

**Note:** This jupyter notebook was executed in Google Colab environment

[Google Colab notebook](https://colab.research.google.com/drive/1-2wo8pIRlmqqN5YW0sMs0FP4jb5oVJ23)

## Download and unpack initial archive

The original TACO dataset is available through the official GitHub repository.
For convenience and reproducibility, this notebook downloads a prepared archive containing the dataset files in the expected folder structure.

The archive is used only to simplify notebook execution in Colab / remote GPU environments. The annotation format remains the original TACO `annotations.json`.

In [1]:
!gdown 1CUxA2TcZFhVJWvNH4qwlOSGDPlxodRPF
!unzip taco.zip -d /content

Downloading...
From (original): https://drive.google.com/uc?id=1CUxA2TcZFhVJWvNH4qwlOSGDPlxodRPF
From (redirected): https://drive.google.com/uc?id=1CUxA2TcZFhVJWvNH4qwlOSGDPlxodRPF&confirm=t&uuid=c51703ed-5783-4aaa-ad3b-09178ed86f2f
To: /content/taco.zip
 31% 1.62G/5.23G [00:17<01:14, 48.3MB/s]Traceback (most recent call last):
  File "/usr/local/bin/gdown", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gdown/__main__.py", line 171, in main
    download(
  File "/usr/local/lib/python3.12/dist-packages/gdown/download.py", line 376, in download
    f.write(chunk)
KeyboardInterrupt
 31% 1.62G/5.23G [00:19<00:42, 84.0MB/s]
^C
unzip:  cannot find or open taco.zip, taco.zip.zip or taco.zip.ZIP.


## Import libraries

In [3]:
import os
import pandas as pd
import ast
import json

## Set environment paths

Based on given paths, I set the appropriate path values

In [ ]:
def is_google_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


IN_COLAB = is_google_colab()

if IN_COLAB:
    ROOT_PATH = "/content"
    DATAFRAME_PATH = "/content/drive/MyDrive/taco_trash"
else:
    ROOT_PATH = "/home/ubuntu"
    DATAFRAME_PATH = os.path.join(ROOT_PATH, "taco_trash")

TACO_CLASSIFICATION_PATH = os.path.join(ROOT_PATH, "taco")

print("IN_COLAB:", IN_COLAB)
print("ROOT_PATH:", ROOT_PATH)
print("DATAFRAME_PATH:", DATAFRAME_PATH)
print("TACO_CLASSIFICATION_PATH:", TACO_CLASSIFICATION_PATH)

IN_COLAB: True
ROOT_PATH: /content
DATAFRAME_PATH: /content/drive/MyDrive/taco_trash
TACO_CLASSIFICATION_PATH: /content/taco


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

## Thundercompute setup

In case of running notebook in thundercompute environment, I recommend to use [rclone](https://rclone.org/) utility tool

It allows data sync with multiple providers, including Google Drive

[The guide](https://rclone.org/drive/) about configuring sync between rclone and google drive

In [ ]:
if not IN_COLAB:
    !curl https://rclone.org/install.sh | sudo bash

## Read `annotations.json`

In [18]:
ann_path = os.path.join(TACO_CLASSIFICATION_PATH, "annotations.json")
with open(ann_path, "r") as f:
    data = json.load(f)

print(data.keys())

for key in data.keys():
    if isinstance(data[key], list):
        print(key, len(data[key]))

dict_keys(['info', 'images', 'annotations', 'scene_annotations', 'licenses', 'categories', 'scene_categories'])
images 1500
annotations 4784
scene_annotations 4296
licenses 0
categories 60
scene_categories 7


The annotation file follows a COCO-like structure:

- `images`: image metadata and file paths;
- `annotations`: object-level annotations with bbox and segmentation;
- `categories`: object category definitions;
- `scene_annotations`: image-level scene labels;
- `scene_categories`: scene category definitions.

For detection and segmentation, the most important tables are `images`, `annotations`, and `categories`.

## Build dataframes

In [8]:
df_images = pd.DataFrame(data["images"])
df_annotations = pd.DataFrame(data["annotations"])
df_categories = pd.DataFrame(data["categories"])
df_scene_annotations = pd.DataFrame(data["scene_annotations"])
df_scene_categories = pd.DataFrame(data["scene_categories"])

In [9]:
def parse_bbox(v):
    if isinstance(v, str):
        return ast.literal_eval(v)
    return v

df = df_annotations.merge(
    df_images,
    left_on="image_id",
    right_on="id",
    how="left",
    suffixes=("", "_img"),
).merge(
    df_categories,
    left_on="category_id",
    right_on="id",
    how="left",
    suffixes=("", "_cat"),
).rename(
    columns={
        "id_x": "annotation_id",
        "id_y": "category_id_full",
        "name": "category_name",
    }
)
df['file_path'] = df['file_name'].map(lambda x: os.path.join(TACO_CLASSIFICATION_PATH, x) )

In [ ]:
# Dataframe integrity checks

print("Images:", len(df_images))
print("Annotations:", len(df_annotations))
print("Categories:", len(df_categories))
print("Merged dataframe:", len(df))

assert len(df) == len(df_annotations), "Some annotations were lost during merge"
assert df["file_path"].notna().all(), "Some annotations have missing file paths"
assert df["category_name"].notna().all(), "Some annotations have missing category names"

missing_files = df[~df["file_path"].map(os.path.exists)]
print("Missing image files:", len(missing_files))

if len(missing_files) > 0:
    display(missing_files[["image_id", "file_name", "file_path"]].head())

In [13]:
df.head(10)

,id,image_id,category_id,segmentation,area,bbox,iscrowd,id_img,width,height,file_name,license,flickr_url,coco_url,date_captured,flickr_640_url,supercategory,id_cat,category_name,file_path
0,1,0,6,"[[561.0, 1238.0, 568.0, 1201.0, 567.0, 1175.0,...",403954.0,"[517.0, 127.0, 447.0, 1322.0]",0,0,1537,2049,batch_1/000006.jpg,None,https://farm66.staticflickr.com/65535/33978196...,None,None,https://farm66.staticflickr.com/65535/33978196...,Bottle,6,Glass bottle,/content/taco/batch_1/000006.jpg
1,2,1,18,"[[928.0, 1876.0, 938.0, 1856.0, 968.0, 1826.0,...",1071259.5,"[1.0, 457.0, 1429.0, 1519.0]",0,1,1537,2049,batch_1/000008.jpg,None,https://farm66.staticflickr.com/65535/47803331...,None,None,https://farm66.staticflickr.com/65535/47803331...,Carton,18,Meal carton,/content/taco/batch_1/000008.jpg
2,3,1,14,"[[617.0, 383.0, 703.0, 437.0, 713.0, 456.0, 72...",99583.5,"[531.0, 292.0, 1006.0, 672.0]",0,1,1537,2049,batch_1/000008.jpg,None,https://farm66.staticflickr.com/65535/47803331...,None,None,https://farm66.staticflickr.com/65535/47803331...,Carton,14,Other carton,/content/taco/batch_1/000008.jpg
3,4,2,5,"[[670.0, 993.0, 679.0, 998.0, 684.0, 1001.0, 6...",73832.5,"[632.0, 987.0, 500.0, 374.0]",0,2,1537,2049,batch_1/000010.jpg,None,https://farm66.staticflickr.com/65535/40888872...,None,None,https://farm66.staticflickr.com/65535/40888872...,Bottle,5,Clear plastic bottle,/content/taco/batch_1/000010.jpg
4,5,2,7,"[[647.0, 1028.0, 650.0, 1022.0, 653.0, 1016.0,...",915.0,"[632.0, 989.0, 44.0, 51.0]",0,2,1537,2049,batch_1/000010.jpg,None,https://farm66.staticflickr.com/65535/40888872...,None,None,https://farm66.staticflickr.com/65535/40888872...,Bottle cap,7,Plastic bottle cap,/content/taco/batch_1/000010.jpg
5,6,3,5,"[[354.0, 1268.0, 351.0, 1252.0, 347.0, 1237.0,...",98379.5,"[209.0, 920.0, 454.0, 562.0]",0,3,2049,1537,batch_1/000019.jpg,None,https://farm66.staticflickr.com/65535/47803331...,None,None,https://farm66.staticflickr.com/65535/47803331...,Bottle,5,Clear plastic bottle,/content/taco/batch_1/000019.jpg
6,7,3,5,"[[1239.0, 841.0, 1247.0, 842.0, 1252.0, 835.0,...",43678.0,"[1212.0, 822.0, 179.0, 446.0]",0,3,2049,1537,batch_1/000019.jpg,None,https://farm66.staticflickr.com/65535/47803331...,None,None,https://farm66.staticflickr.com/65535/47803331...,Bottle,5,Clear plastic bottle,/content/taco/batch_1/000019.jpg
7,8,3,7,"[[643.0, 1453.0, 649.0, 1445.0, 653.0, 1442.0,...",578.5,"[634.0, 1442.0, 29.0, 39.0]",0,3,2049,1537,batch_1/000019.jpg,None,https://farm66.staticflickr.com/65535/47803331...,None,None,https://farm66.staticflickr.com/65535/47803331...,Bottle cap,7,Plastic bottle cap,/content/taco/batch_1/000019.jpg
8,9,3,12,"[[730.0, 852.0, 700.0, 810.0, 686.0, 789.0, 68...",61402.0,"[589.0, 548.0, 341.0, 405.0]",0,3,2049,1537,batch_1/000019.jpg,None,https://farm66.staticflickr.com/65535/47803331...,None,None,https://farm66.staticflickr.com/65535/47803331...,Can,12,Drink can,/content/taco/batch_1/000019.jpg
9,10,4,12,"[[1032.0, 1170.0, 1040.0, 1144.0, 1046.0, 1125...",93266.0,"[556.0, 944.0, 490.0, 336.0]",0,4,1537,2049,batch_1/000026.jpg,None,https://farm66.staticflickr.com/65535/33978199...,None,None,https://farm66.staticflickr.com/65535/33978199...,Can,12,Drink can,/content/taco/batch_1/000026.jpg


The main output is `annotations.csv`. Each row represents one annotated object, not one image.

This is important because a single image can contain multiple litter instances.

The dataframe includes:
  - image metadata;
  - annotation id;
  - category and supercategory;
  - bounding box;
  - segmentation polygon;
  - resolved local image path;
  - bbox size statistics.

Later notebooks use this dataframe to build PyTorch datasets, train detection models, and run diagnostic
experiments.

## Save dataframe

Save given dataframe based on the environment it is run

### Google Colab

In [ ]:
os.makedirs(DATAFRAME_PATH, exist_ok=True)

df.to_csv(os.path.join(DATAFRAME_PATH, "annotations_processed.csv"), index=False)

df_images.to_csv(os.path.join(DATAFRAME_PATH, "images.csv"), index=False)
df_annotations.to_csv(os.path.join(DATAFRAME_PATH, "annotations_raw.csv"), index=False)
df_categories.to_csv(os.path.join(DATAFRAME_PATH, "categories.csv"), index=False)
df_scene_annotations.to_csv(os.path.join(DATAFRAME_PATH, "scene_annotations.csv"), index=False)
df_scene_categories.to_csv(os.path.join(DATAFRAME_PATH, "scene_categories.csv"), index=False)

### Thunder compute

In [ ]:
!rclone copy ./taco_trash gdrive:taco_trash --progress